# 02 — Análisis Coyuntural y Modelación Macroeconómica (SARIMA)
Este notebook ilustra el flujo analítico completo exigido para la variable objetivo del proyecto:
1. **Lectura y Coyuntura:** Extracción desde SQLite y cálculo de la variación interanual (12 meses).
2. **Modelación Econométrica:** Estimación de un modelo estacional SARIMA.
3. **Evaluación Fuera de Muestra (Out-of-Sample):** Validación recursiva a $H=12$ meses comparando contra un *benchmark* ingenuo (Caminata Aleatoria).

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sqlalchemy import text
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Silenciar advertencias de convergencia en el bucle de pronóstico
warnings.filterwarnings("ignore")

# Configuración de rutas
sys.path.append(str(Path.cwd().parent / "src"))
import carga_db

# Estilo visual de gráficos
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

In [ ]:
engine = carga_db.obtener_engine()

# Seleccionar la serie objetivo: Imacec (o IPC si se prefiere)
query_serie = text("""
SELECT id_serie, nombre, unidad 
FROM series 
WHERE id_serie LIKE '%IMC%' OR nombre LIKE '%Imacec%'
LIMIT 1
""")

with engine.connect() as conn:
    meta = pd.read_sql(query_serie, conn)

if meta.empty:
    # Fallback a la primera serie mensual si no encuentra el patrón exacto
    with engine.connect() as conn:
        meta = pd.read_sql(text("SELECT id_serie, nombre, unidad FROM series WHERE frecuencia = 'M' LIMIT 1"), conn)

ID_SERIE = meta.iloc[0]["id_serie"]
NOMBRE = meta.iloc[0]["nombre"]
UNIDAD = meta.iloc[0]["unidad"]

print(f"Variable Objetivo Seleccionada: {NOMBRE} ({ID_SERIE})")

In [ ]:
# Consulta de la serie histórica desde la tabla 'observaciones'
query_obs = text("""
SELECT fecha, valor 
FROM observaciones 
WHERE id_serie = :id 
ORDER BY fecha
""")

with engine.connect() as conn:
    df = pd.read_sql(query_obs, conn, params={"id": ID_SERIE}, parse_dates=["fecha"])

df = df.set_index("fecha").asfreq("MS")

# Variación interanual (% a 12 meses)
df["var_ia"] = 100.0 * (df["valor"] / df["valor"].shift(12) - 1.0)

# Gráfico de Coyuntura
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(df.index, df["valor"], color="#1f77b4", lw=2, label="Nivel")
axes[0].set_title(f"Coyuntura Macroeconómica: {NOMBRE}", fontsize=12, fontweight="bold")
axes[0].set_ylabel(UNIDAD)
axes[0].legend(loc="upper left")

axes[1].plot(df.index, df["var_ia"], color="#d62728", lw=1.5, label="Var. % Interanual (12M)")
axes[1].axhline(0, color="black", linestyle="--", linewidth=0.8)
axes[1].set_title("Variación Interanual (%)", fontsize=11)
axes[1].set_ylabel("%")
axes[1].legend(loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
# Serie limpia para modelación
y = df["valor"].dropna()

# Estimación del modelo SARIMA(1, 1, 1)(1, 0, 1, 12)
modelo = SARIMAX(y, order=(1, 1, 1), seasonal_order=(1, 0, 1, 12))
resultado = modelo.fit(disp=False)

print("=== RESUMEN DE ESTIMACIÓN DEL MODELO SARIMA ===")
print(resultado.summary().tables[0])
print(resultado.summary().tables[1])

In [ ]:
# Horizonte de evaluación fuera de muestra (últimos 12 meses)
H = 12 
entrenamiento, prueba = y.iloc[:-H], y.iloc[-H:]

pronosticos = []
historia = entrenamiento.copy()

# Bucle de pronóstico recursivo 1-paso-adelante (sin look-ahead bias)
for fecha in prueba.index:
    m = SARIMAX(historia, order=(1, 1, 1), seasonal_order=(1, 0, 1, 12))
    r = m.fit(disp=False)
    pred_1p = r.forecast(1).iloc[0]
    pronosticos.append(pred_1p)
    # Actualización del conjunto de datos con la observación real
    historia = pd.concat([historia, prueba.loc[[fecha]]])

pronosticos = pd.Series(pronosticos, index=prueba.index)
benchmark_ingenuo = y.shift(1).loc[prueba.index]  # Caminata aleatoria / valor previo

# Funciones de métricas de error
def rmse(obs, pred):
    return float(np.sqrt(np.mean((obs - pred) ** 2)))

def mae(obs, pred):
    return float(np.mean(np.abs(obs - pred)))

# Resultados de precisión
print(f"=== EVALUACIÓN DE PRONÓSTICO ({H} MESES FUERA DE MUESTRA) ===")
print(f"SARIMA           -> RMSE: {rmse(prueba, pronosticos):.3f} | MAE: {mae(prueba, pronosticos):.3f}")
print(f"Benchmark Ingenuo -> RMSE: {rmse(prueba, benchmark_ingenuo):.3f} | MAE: {mae(prueba, benchmark_ingenuo):.3f}")

# Gráfico comparativo de pronósticos
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(y.iloc[-36:], label="Observado Real", color="black", lw=2)
ax.plot(pronosticos, "--o", label="SARIMA (Out-of-sample)", color="#1f77b4", ms=4)
ax.plot(benchmark_ingenuo, ":s", label="Benchmark Ingenuo", color="#ff7f0e", ms=3)

ax.set_title(f"Evaluación de Pronóstico Fuera de Muestra: {NOMBRE}", fontsize=12, fontweight="bold")
ax.set_ylabel(UNIDAD)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()